# 10. Hyperparameter Search & Sensitivity Analysis

**Purpose**: Find optimal hyperparameters for the three key learnable models 
(XGBoost, LSTM, LambdaMART) and validate sensitivity of other components 
(IRT, GraphSAGE, K-Means, ALS).

**Output**: Best parameters saved to `configs/config.yaml` and full results 
logged to `results/hp_search/`.

| Model | Search Method | Budget | Metric |
|-------|--------------|--------|--------|
| XGBoost (Confidence) | RandomizedSearchCV | 50 iter, 5-fold | F1-macro |
| LSTM (Prediction) | Random Search | 20 configs | AUC-ROC macro |
| LambdaMART (Recommendation) | Grid Search | 81 configs | NDCG@10 |
| IRT | Compare 1PL/2PL/3PL | 3 models | Pearson r(theta, accuracy) |
| GraphSAGE | Grid | 9 configs | Link prediction AUC |
| K-Means | Silhouette | K=3..8 | Silhouette score |
| ALS | Grid | 3 configs | Recall@10 |

In [ ]:
import sys, os, json, time, logging, warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import seaborn as sns

# Project root
ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from agents.utils import set_global_seed, load_config

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s: %(message)s')
warnings.filterwarnings('ignore')

config = load_config('configs/config.yaml')
SEED = config['global']['seed']
set_global_seed(SEED)

HP_DIR = Path('results/hp_search')
HP_DIR.mkdir(parents=True, exist_ok=True)

print(f'Seed: {SEED}, Project root: {ROOT}')

## 0. Load Data

In [ ]:
# Load preprocessed data (run pipeline first if not available)
splits_dir = Path('data/splits')

if (splits_dir / 'train.parquet').exists():
    train_df = pd.read_parquet(splits_dir / 'train.parquet')
    val_df = pd.read_parquet(splits_dir / 'val.parquet')
    test_df = pd.read_parquet(splits_dir / 'test.parquet')
    print(f'Train: {len(train_df):,} rows, Val: {len(val_df):,}, Test: {len(test_df):,}')
    print(f'Users: train={train_df["user_id"].nunique()}, val={val_df["user_id"].nunique()}')
else:
    print('ERROR: No preprocessed data found. Run the data pipeline first.')
    print('  from data.loader import EdNetLoader')
    print('  from data.preprocessor import EdNetPreprocessor')
    print('  loader = EdNetLoader(); interactions = loader.load_interactions()')
    print('  preprocessor = EdNetPreprocessor(); splits = preprocessor.run(interactions, chunked=True)')

---
## 1. XGBoost Hyperparameter Search (ConfidenceAgent)

RandomizedSearchCV with 50 iterations, 5-fold stratified CV, scoring=F1-macro.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

from agents.confidence_agent import ConfidenceAgent

# Build features and labels using ConfidenceAgent internals
agent_conf = ConfidenceAgent()
df_conf = train_df.dropna(subset=['elapsed_time', 'correct', 'changed_answer']).copy()

labels = agent_conf._assign_labels(df_conf)
X_conf = agent_conf._build_features(df_conf)
y_conf = labels.values

print(f'XGBoost search data: {X_conf.shape[0]:,} samples, {X_conf.shape[1]} features')
print(f'Label distribution:\n{pd.Series(y_conf).value_counts().sort_index()}')

In [ ]:
%%time

param_distributions = {
    'max_depth': [4, 5, 6, 7, 8],
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
}

base_xgb = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=6,
    eval_metric='mlogloss',
    verbosity=0,
    random_state=SEED,
)

search_xgb = RandomizedSearchCV(
    base_xgb,
    param_distributions=param_distributions,
    n_iter=50,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='f1_macro',
    n_jobs=-1,
    random_state=SEED,
    verbose=1,
    return_train_score=True,
)

search_xgb.fit(X_conf.values.astype(np.float32), y_conf)

print(f'\nBest F1-macro: {search_xgb.best_score_:.4f}')
print(f'Best params: {search_xgb.best_params_}')

In [ ]:
# Save XGBoost results
xgb_results = pd.DataFrame(search_xgb.cv_results_)
xgb_results.to_csv(HP_DIR / 'xgboost_search_results.csv', index=False)

xgb_best = {
    'best_score': round(search_xgb.best_score_, 4),
    'best_params': search_xgb.best_params_,
}
with open(HP_DIR / 'xgboost_best.json', 'w') as f:
    json.dump(xgb_best, f, indent=2, default=str)

# Validate on val set
df_val_conf = val_df.dropna(subset=['elapsed_time', 'correct', 'changed_answer']).copy()
y_val_labels = agent_conf._assign_labels(df_val_conf).values
X_val_conf = agent_conf._build_features(df_val_conf).values.astype(np.float32)
y_val_pred = search_xgb.best_estimator_.predict(X_val_conf)
val_f1 = f1_score(y_val_labels, y_val_pred, average='macro', zero_division=0)
print(f'Validation F1-macro: {val_f1:.4f}')
print(classification_report(y_val_labels, y_val_pred, zero_division=0))

In [ ]:
# Visualize: parameter importance heatmap
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, param in zip(axes.flat, param_distributions.keys()):
    col = f'param_{param}'
    if col in xgb_results.columns:
        grouped = xgb_results.groupby(col)['mean_test_score'].mean()
        grouped.plot(kind='bar', ax=ax, color='steelblue')
        ax.set_title(param)
        ax.set_ylabel('Mean F1-macro')
        ax.tick_params(axis='x', rotation=0)

plt.suptitle('XGBoost Hyperparameter Impact on F1-macro', fontsize=14)
plt.tight_layout()
plt.savefig(HP_DIR / 'xgboost_param_impact.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. LSTM Hyperparameter Search (PredictionAgent)

Random Search over 20 configurations. Metric: AUC-ROC macro on validation set.

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

from agents.prediction_agent import (
    GapSequenceDataset, GapPredictionLSTM, DEVICE,
    NUM_TAGS, TAG_EMB_DIM, PART_EMB_DIM, CONF_EMB_DIM,
    NUM_PARTS, NUM_CONF_CLASSES, SCALAR_FEATURES,
)

# LSTM search grid
lstm_grid = {
    'hidden_dim': [64, 128, 256],
    'num_layers': [1, 2, 3],
    'dropout': [0.1, 0.2, 0.3, 0.5],
    'learning_rate': [5e-4, 1e-3, 2e-3],
}

# Total combos: 3*3*4*3 = 108 -> sample 20
all_combos = list(product(*lstm_grid.values()))
rng = np.random.RandomState(SEED)
sampled_idx = rng.choice(len(all_combos), size=min(20, len(all_combos)), replace=False)
sampled_configs = [dict(zip(lstm_grid.keys(), all_combos[i])) for i in sampled_idx]

print(f'LSTM: {len(all_combos)} total combos, sampling {len(sampled_configs)}')
for i, cfg in enumerate(sampled_configs):
    print(f'  [{i+1}] {cfg}')

In [ ]:
%%time

# Build datasets once
train_dataset = GapSequenceDataset(train_df)
val_dataset = GapSequenceDataset(val_df)
print(f'Train sequences: {len(train_dataset)}, Val sequences: {len(val_dataset)}')

def train_lstm_config(cfg, train_ds, val_ds, max_epochs=30, patience=5):
    """Train one LSTM config and return val AUC-ROC."""
    set_global_seed(SEED)
    
    # Build model with custom config
    input_dim = TAG_EMB_DIM + SCALAR_FEATURES + PART_EMB_DIM + CONF_EMB_DIM
    
    class CustomLSTM(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.tag_emb = torch.nn.Embedding(NUM_TAGS, TAG_EMB_DIM, padding_idx=0)
            self.part_emb = torch.nn.Embedding(NUM_PARTS, PART_EMB_DIM)
            self.conf_emb = torch.nn.Embedding(NUM_CONF_CLASSES, CONF_EMB_DIM)
            self.lstm = torch.nn.LSTM(
                input_size=input_dim,
                hidden_size=cfg['hidden_dim'],
                num_layers=cfg['num_layers'],
                dropout=cfg['dropout'] if cfg['num_layers'] > 1 else 0,
                batch_first=True,
            )
            self.fc = torch.nn.Linear(cfg['hidden_dim'], NUM_TAGS)
        
        def forward(self, x):
            tag_ids = x[:,:,0].long().clamp(0, NUM_TAGS-1)
            correct = x[:,:,1:2]
            elapsed = x[:,:,2:3]
            changed = x[:,:,3:4]
            part_ids = x[:,:,4].long().clamp(0, NUM_PARTS-1)
            conf_ids = x[:,:,5].long().clamp(0, NUM_CONF_CLASSES-1)
            combined = torch.cat([
                self.tag_emb(tag_ids), correct, elapsed, changed,
                self.part_emb(part_ids), self.conf_emb(conf_ids)
            ], dim=-1)
            out, _ = self.lstm(combined)
            return self.fc(out[:, -1, :])
    
    model = CustomLSTM().to(DEVICE)
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['learning_rate'])
    
    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, num_workers=0)
    
    best_val_loss = float('inf')
    no_improve = 0
    
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_losses.append(criterion(model(xb), yb).item())
        
        vl = np.mean(val_losses)
        if vl < best_val_loss:
            best_val_loss = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break
    
    # Compute val AUC
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            all_preds.append(torch.sigmoid(model(xb)).cpu().numpy())
            all_labels.append(yb.numpy())
    
    y_true = np.concatenate(all_labels)
    y_score = np.concatenate(all_preds)
    mask = y_true.sum(axis=0) > 0
    
    try:
        auc = roc_auc_score(y_true[:, mask], y_score[:, mask], average='macro')
    except ValueError:
        auc = 0.0
    
    n_params = sum(p.numel() for p in model.parameters())
    return {'val_auc': round(auc, 4), 'val_loss': round(best_val_loss, 4), 'n_params': n_params}

# Run search
lstm_results = []
for i, cfg in enumerate(sampled_configs):
    t0 = time.time()
    metrics = train_lstm_config(cfg, train_dataset, val_dataset)
    elapsed = time.time() - t0
    result = {**cfg, **metrics, 'time_s': round(elapsed, 1)}
    lstm_results.append(result)
    print(f'[{i+1}/{len(sampled_configs)}] {cfg} -> AUC={metrics["val_auc"]:.4f} ({elapsed:.0f}s)')

lstm_df = pd.DataFrame(lstm_results).sort_values('val_auc', ascending=False)
lstm_df.to_csv(HP_DIR / 'lstm_search_results.csv', index=False)

print(f'\nBest LSTM config:')
print(lstm_df.iloc[0])

In [ ]:
# Visualize LSTM results
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, param in zip(axes, ['hidden_dim', 'num_layers', 'dropout', 'learning_rate']):
    grouped = lstm_df.groupby(param)['val_auc'].mean()
    grouped.plot(kind='bar', ax=ax, color='coral')
    ax.set_title(param)
    ax.set_ylabel('Mean Val AUC-ROC')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('LSTM Hyperparameter Impact on AUC-ROC', fontsize=14)
plt.tight_layout()
plt.savefig(HP_DIR / 'lstm_param_impact.png', dpi=150, bbox_inches='tight')
plt.show()

# Save best
best_lstm = lstm_df.iloc[0].to_dict()
with open(HP_DIR / 'lstm_best.json', 'w') as f:
    json.dump(best_lstm, f, indent=2, default=str)

---
## 3. LambdaMART Hyperparameter Search (RecommendationAgent)

Grid search over num_leaves, min_child_samples, learning_rate, n_estimators. 
Metric: NDCG@10 on validation set.

In [ ]:
import lightgbm as lgb
from sklearn.metrics import ndcg_score

lambdamart_grid = {
    'num_leaves': [15, 31, 63],
    'min_child_samples': [5, 10, 20],
    'learning_rate': [0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
}

all_lm_combos = list(product(*lambdamart_grid.values()))
print(f'LambdaMART: {len(all_lm_combos)} total configs to evaluate')
print('Note: This requires pre-built ranking features from RecommendationAgent.')
print('If ranking features are not available, this section will use a simplified proxy.')

In [ ]:
# Check if ranking features exist, otherwise create proxy data
ranking_path = Path('results/ranking_features_train.parquet')

if ranking_path.exists():
    rank_train = pd.read_parquet(ranking_path)
    rank_val = pd.read_parquet('results/ranking_features_val.parquet')
    print(f'Loaded ranking features: train={len(rank_train):,}, val={len(rank_val):,}')
else:
    print('Ranking features not found. Building proxy from interaction data...')
    print('For full LambdaMART search, first run RecommendationAgent.build_ranking_features()')
    print('\nUsing simplified pointwise proxy for parameter sensitivity.')
    
    # Proxy: predict correct/incorrect as relevance signal
    # This tests LightGBM parameter sensitivity, not full ranking quality
    from agents.confidence_agent import FEATURE_NAMES as CONF_FEATURES
    rank_train = None  # Will skip full LambdaMART search

In [ ]:
if rank_train is not None:
    lm_results = []
    
    # Determine feature/label columns
    feature_cols = [c for c in rank_train.columns if c not in ('user_id', 'item_id', 'relevance', 'group')]
    
    X_train_lm = rank_train[feature_cols].values
    y_train_lm = rank_train['relevance'].values
    groups_train = rank_train.groupby('user_id').size().values
    
    X_val_lm = rank_val[feature_cols].values
    y_val_lm = rank_val['relevance'].values
    groups_val = rank_val.groupby('user_id').size().values
    
    for i, combo in enumerate(all_lm_combos):
        cfg = dict(zip(lambdamart_grid.keys(), combo))
        t0 = time.time()
        
        ranker = lgb.LGBMRanker(
            objective='lambdarank',
            metric='ndcg',
            ndcg_eval_at=[10],
            **cfg,
            verbose=-1,
            random_state=SEED,
        )
        ranker.fit(
            X_train_lm, y_train_lm, group=groups_train,
            eval_set=[(X_val_lm, y_val_lm)], eval_group=[groups_val],
            callbacks=[lgb.log_evaluation(0)],
        )
        
        # Compute NDCG@10 per user on val
        val_preds = ranker.predict(X_val_lm)
        # Group predictions by user for NDCG
        ndcg_scores = []
        idx = 0
        for g in groups_val:
            if g >= 2:
                y_true_g = y_val_lm[idx:idx+g].reshape(1, -1)
                y_pred_g = val_preds[idx:idx+g].reshape(1, -1)
                try:
                    ndcg_scores.append(ndcg_score(y_true_g, y_pred_g, k=10))
                except ValueError:
                    pass
            idx += g
        
        avg_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0.0
        elapsed = time.time() - t0
        result = {**cfg, 'ndcg_at_10': round(avg_ndcg, 4), 'time_s': round(elapsed, 1)}
        lm_results.append(result)
        
        if (i + 1) % 10 == 0:
            print(f'  [{i+1}/{len(all_lm_combos)}] best so far: NDCG@10={max(r["ndcg_at_10"] for r in lm_results):.4f}')
    
    lm_df = pd.DataFrame(lm_results).sort_values('ndcg_at_10', ascending=False)
    lm_df.to_csv(HP_DIR / 'lambdamart_search_results.csv', index=False)
    
    print(f'\nBest LambdaMART config:')
    print(lm_df.iloc[0])
    
    with open(HP_DIR / 'lambdamart_best.json', 'w') as f:
        json.dump(lm_df.iloc[0].to_dict(), f, indent=2, default=str)
else:
    print('Skipping full LambdaMART search (no ranking features).')
    print('Run RecommendationAgent pipeline first, then re-run this cell.')

---
## 4. Sensitivity Analysis

### 4.1 IRT: 1PL vs 2PL vs 3PL

In [ ]:
from agents.diagnostic_agent import DiagnosticAgent
from scipy.stats import pearsonr

# Combine train+val for IRT calibration
irt_df = pd.concat([train_df, val_df], ignore_index=True)

# Raw per-user accuracy for correlation
user_accuracy = irt_df.groupby('user_id')['correct'].mean()

irt_results = []
for model_type in ['1PL', '2PL', '3PL']:
    t0 = time.time()
    diag = DiagnosticAgent()
    params = diag.calibrate_from_interactions(irt_df, min_answers_per_q=20)
    
    if model_type == '1PL':
        # Fix a=1 for all items (Rasch model)
        params.a[:] = 1.0
        params.c[:] = 0.0
    elif model_type == '2PL':
        # Keep estimated a, fix c=0
        params.c[:] = 0.0
    # 3PL: use all params as-is
    
    diag.irt_params = params
    
    # Estimate theta for each user
    thetas = {}
    for uid, grp in irt_df.groupby('user_id'):
        responses = []
        for _, row in grp.head(50).iterrows():  # cap at 50 for speed
            qid = str(row['question_id'])
            idx = diag._qid_to_idx.get(qid)
            if idx is not None:
                responses.append((idx, bool(row['correct'])))
        if responses:
            theta, se = diag.update_theta(responses)
            thetas[uid] = theta
    
    # Compute correlation with raw accuracy
    common = set(thetas.keys()) & set(user_accuracy.index)
    theta_vals = [thetas[u] for u in common]
    acc_vals = [user_accuracy[u] for u in common]
    r, p_val = pearsonr(theta_vals, acc_vals)
    
    elapsed = time.time() - t0
    irt_results.append({
        'model': model_type,
        'pearson_r': round(r, 4),
        'p_value': round(p_val, 6),
        'n_users': len(common),
        'n_items': len(params),
        'theta_mean': round(np.mean(theta_vals), 3),
        'theta_std': round(np.std(theta_vals), 3),
        'time_s': round(elapsed, 1),
    })
    print(f'{model_type}: r={r:.4f}, p={p_val:.2e}, theta: {np.mean(theta_vals):.3f} +/- {np.std(theta_vals):.3f}')

irt_sens_df = pd.DataFrame(irt_results)
irt_sens_df.to_csv(HP_DIR / 'irt_sensitivity.csv', index=False)
display(irt_sens_df)

### 4.2 GraphSAGE: hidden_dim x layers

In [ ]:
from agents.kg_agent import KnowledgeGraphAgent, GraphSAGEModel
from data.loader import EdNetLoader

# Load metadata for graph building
loader = EdNetLoader(data_dir='data/raw')
questions_df = loader.questions
lectures_df = loader.lectures

# GraphSAGE sensitivity: hidden x layers
gs_hidden_dims = [64, 128, 256]
# Note: current GraphSAGEModel is fixed 2-layer. We test hidden_dim sensitivity.
# For layer sensitivity, we'd need to modify the model class.

gs_results = []
for hidden in gs_hidden_dims:
    t0 = time.time()
    kg = KnowledgeGraphAgent()
    kg._gs_hidden = hidden
    kg.build_graph(questions_df, lectures_df)
    
    # Build prerequisites and train embeddings
    try:
        kg.build_prerequisites(train_df)
        # If train_embeddings exists, use it
        if hasattr(kg, 'train_embeddings'):
            metrics = kg.train_embeddings()
            auc = metrics.get('test_auc', 0.0)
        else:
            auc = 0.0
            print(f'  hidden={hidden}: train_embeddings not available, skipping AUC')
    except Exception as e:
        auc = 0.0
        print(f'  hidden={hidden}: error: {e}')
    
    elapsed = time.time() - t0
    gs_results.append({
        'hidden_dim': hidden,
        'link_pred_auc': round(auc, 4),
        'n_nodes': kg.graph.number_of_nodes(),
        'n_edges': kg.graph.number_of_edges(),
        'time_s': round(elapsed, 1),
    })
    print(f'GraphSAGE hidden={hidden}: AUC={auc:.4f} ({elapsed:.0f}s)')

gs_df = pd.DataFrame(gs_results)
gs_df.to_csv(HP_DIR / 'graphsage_sensitivity.csv', index=False)
display(gs_df)

### 4.3 K-Means: Silhouette validation

In [ ]:
from agents.personalization_agent import PersonalizationAgent, extract_user_features
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Extract features
user_feats = extract_user_features(train_df)
print(f'User features: {user_feats.shape}')

# Run K-Means for K=3..8
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from agents.personalization_agent import FEATURE_NAMES

X_raw = user_feats[FEATURE_NAMES].values.astype(np.float64)
X_raw = np.nan_to_num(X_raw, nan=0.0, posinf=5.0, neginf=-5.0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

km_results = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=300)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    ch = calinski_harabasz_score(X_scaled, labels)
    db = davies_bouldin_score(X_scaled, labels)
    
    km_results.append({
        'k': k, 'silhouette': round(sil, 4),
        'calinski_harabasz': round(ch, 1),
        'davies_bouldin': round(db, 4),
        'inertia': round(km.inertia_, 1),
    })
    print(f'K={k}: silhouette={sil:.4f}, CH={ch:.1f}, DB={db:.4f}')

km_df = pd.DataFrame(km_results)
km_df.to_csv(HP_DIR / 'kmeans_sensitivity.csv', index=False)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, label in zip(axes, ['silhouette', 'calinski_harabasz', 'davies_bouldin'],
                              ['Silhouette (higher=better)', 'Calinski-Harabasz (higher=better)', 'Davies-Bouldin (lower=better)']):
    ax.plot(km_df['k'], km_df[metric], 'o-', color='steelblue', linewidth=2)
    best_k = km_df.loc[km_df[metric].idxmax() if 'bouldin' not in metric else km_df[metric].idxmin(), 'k']
    ax.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'Best K={best_k}')
    ax.set_xlabel('K'); ax.set_ylabel(label); ax.legend()

plt.suptitle('K-Means Cluster Validation', fontsize=14)
plt.tight_layout()
plt.savefig(HP_DIR / 'kmeans_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 ALS: n_factors sensitivity

In [ ]:
from sklearn.decomposition import TruncatedSVD
import scipy.sparse as sp

# Build user-tag accuracy matrix
if 'tags' in train_df.columns:
    tag_df = train_df[['user_id', 'tags', 'correct']].copy()
    tag_df['tags'] = tag_df['tags'].apply(
        lambda x: x if isinstance(x, list) else 
        [int(t) for t in str(x).split(';') if t.strip().isdigit()]
    )
    exploded = tag_df.explode('tags').dropna(subset=['tags'])
    exploded['tags'] = exploded['tags'].astype(int)
    
    user_tag_acc = exploded.groupby(['user_id', 'tags'])['correct'].mean().reset_index()
    user_ids = sorted(user_tag_acc['user_id'].unique())
    tag_ids = sorted(user_tag_acc['tags'].unique())
    
    user_map = {u: i for i, u in enumerate(user_ids)}
    tag_map = {t: i for i, t in enumerate(tag_ids)}
    
    rows = user_tag_acc['user_id'].map(user_map).values
    cols = user_tag_acc['tags'].map(tag_map).values
    vals = user_tag_acc['correct'].values
    
    R = sp.csr_matrix((vals, (rows, cols)), shape=(len(user_ids), len(tag_ids)))
    print(f'User-tag matrix: {R.shape}, nnz={R.nnz}')
    
    als_results = []
    for n_factors in [32, 64, 128]:
        svd = TruncatedSVD(n_components=min(n_factors, min(R.shape) - 1), random_state=SEED)
        user_factors = svd.fit_transform(R)
        tag_factors = svd.components_.T
        
        explained_var = svd.explained_variance_ratio_.sum()
        reconstruction = user_factors @ tag_factors.T
        rmse = np.sqrt(np.mean((R.toarray() - reconstruction) ** 2))
        
        als_results.append({
            'n_factors': n_factors,
            'explained_variance': round(explained_var, 4),
            'rmse': round(rmse, 4),
        })
        print(f'n_factors={n_factors}: explained_var={explained_var:.4f}, RMSE={rmse:.4f}')
    
    als_df = pd.DataFrame(als_results)
    als_df.to_csv(HP_DIR / 'als_sensitivity.csv', index=False)
    display(als_df)
else:
    print('Tags column not found — skipping ALS sensitivity.')

---
## 5. Summary Table (for paper)

Table: Hyperparameter Search Results

In [ ]:
# Build summary table for the paper
summary_rows = []

# XGBoost
if 'search_xgb' in dir():
    for param, val in search_xgb.best_params_.items():
        summary_rows.append({
            'Model': 'XGBoost (Confidence)',
            'Parameter': param,
            'Search Range': str(param_distributions.get(param, '-')),
            'Best Value': val,
            'Metric': f'F1-macro = {search_xgb.best_score_:.4f}',
        })

# LSTM
if 'lstm_df' in dir() and len(lstm_df) > 0:
    best = lstm_df.iloc[0]
    for param in ['hidden_dim', 'num_layers', 'dropout', 'learning_rate']:
        summary_rows.append({
            'Model': 'LSTM (Prediction)',
            'Parameter': param,
            'Search Range': str(lstm_grid.get(param, '-')),
            'Best Value': best[param],
            'Metric': f'AUC-ROC = {best["val_auc"]:.4f}',
        })

# IRT
if 'irt_sens_df' in dir():
    best_irt = irt_sens_df.loc[irt_sens_df['pearson_r'].idxmax()]
    summary_rows.append({
        'Model': 'IRT (Diagnostic)',
        'Parameter': 'model_type',
        'Search Range': '1PL, 2PL, 3PL',
        'Best Value': best_irt['model'],
        'Metric': f'r = {best_irt["pearson_r"]:.4f}',
    })

# K-Means
if 'km_df' in dir():
    best_km = km_df.loc[km_df['silhouette'].idxmax()]
    summary_rows.append({
        'Model': 'K-Means (Personalization)',
        'Parameter': 'K',
        'Search Range': '3, 4, 5, 6, 7, 8',
        'Best Value': int(best_km['k']),
        'Metric': f'Silhouette = {best_km["silhouette"]:.4f}',
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(HP_DIR / 'hp_search_summary.csv', index=False)
display(summary_df)
print('\nSaved to results/hp_search/hp_search_summary.csv')

## 6. Update config.yaml with best parameters

In [ ]:
# Read current config
config_path = Path('configs/config.yaml')
with open(config_path) as f:
    config = yaml.safe_load(f)

updated = False

# Update XGBoost params
if 'search_xgb' in dir():
    bp = search_xgb.best_params_
    config['confidence']['xgboost'].update({
        'max_depth': int(bp['max_depth']),
        'n_estimators': int(bp['n_estimators']),
        'learning_rate': float(bp['learning_rate']),
    })
    updated = True
    print(f'Updated XGBoost: {bp}')

# Update LSTM params
if 'lstm_df' in dir() and len(lstm_df) > 0:
    best = lstm_df.iloc[0]
    config['prediction']['lstm'].update({
        'hidden_dim': int(best['hidden_dim']),
        'num_layers': int(best['num_layers']),
        'dropout': float(best['dropout']),
        'learning_rate': float(best['learning_rate']),
    })
    updated = True
    print(f'Updated LSTM: hidden={best["hidden_dim"]}, layers={best["num_layers"]}, '
          f'dropout={best["dropout"]}, lr={best["learning_rate"]}')

if updated:
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False, allow_unicode=True)
    print(f'\nConfig saved to {config_path}')
else:
    print('No updates to save.')